# 扩展检验：盈余公告、借壳与壳价值

先在同一 kernel 运行数据准备与主结果 notebook。本文件仅把结果保留在内存，不写出额外文件。本次用户允许直接实现，跳过 learning 预审；仍遵循表粒度、唯一主键、真实信息时点和 White HC1 标准误规则。

论文依据：本地《Size and value in China》，正文 §3、Table 1、Fig. 1–2、Appendix A4–A7（PDF 第4–7、18–21页）。论文选择剔除最小30%的依据来自两步证据：133起借壳中约83%来自市值最小的30%股票；Fig. 2估计的平均壳价值约为公司市值的29.5%。前者是被排除股票在截面中的比例，后者是壳价值占市值的比例，含义不同。原文同时报告中国最小股票盈余反应较弱、美国方向相反。

当前财务公告从2007年开始，缺扣非单季度盈余；本地合并表是吸收合并，并非 WIND 借壳133事件；缺 CSRC 审批日期、真实上市主表和美国 CRSP/Compustat。缺输入时只显示覆盖缺口，绝不复制论文数值作为本地估计，也不以首条交易记录、EPS、吸收合并替代所需数据。

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from IPython.display import display

extension_inputs = globals().get('extension_inputs', {})
extension_options = globals().get('extension_options', {})
extension_results = {}
extension_coverage_rows = []

## 标准化输入契约

在主 notebook 配置中给出字典 `extension_inputs`。股票代码为字符串，日期为 datetime64，月份为 Period[M]，收益为小数；市值/盈余币种与量纲一致。输入供应方须保证真实历史版本、完整上市股票池与身份链接；列校验不能证明这些条件。

| 字典键 | 一行与主键 | 必需列和口径 |
|---|---|---|
| cn_earnings / us_earnings | 股票×财季；Stkcd,fiscal_quarter | announcement_date,earnings；首次真实披露，CN 单季度扣非净利润，US Compustat 季度盈余；非累计值 |
| cn_daily / us_daily | 股票×交易日；Stkcd,Date | stock_return；停牌缺日不补零 |
| cn_market / us_market | 市场交易日；Date | market_return；日期定义交易日历 |
| cn_monthly_caps / us_monthly_caps | 股票×市值月；Stkcd,cap_month | market_value；全体上市股票真实月末总市值 |
| cn_monthly_full | 股票×持有月；Stkcd,month | lagged_market_value,stock_return；完整股票池，前月信号 |
| reverse_mergers | 借壳交易；deal_id | Stkcd,deal_date,board_date,approval_date；真实成功借壳及审批日 |
| cn_shell_daily | 全股票×日；Stkcd,Date | market_value,stock_return；全量股票池，市值人民币元，覆盖事件两端 |
| cn_ipo | 股票；Stkcd | listing_date；真实证券主表，包含完整IPO历史 |
| bayesian_log_ml | 先验强度；multiplier | ch3_log_ml,ff3_log_ml；仅接受外部已核验的 Barillas–Shanken 解析边际似然 |

`extension_options` 可设置 `event_day_basis='calendar'`（默认）或 `'trading'`、`reverse_history_start='2007-01-01'`、`earnings_release_after_close=False`、`bayesian_provenance`（完整边际似然来源与实现核验记录）。
论文明确730自然日，但未明说前后60日基准，默认采用自然日并披露；可用交易日口径做敏感性检查。非交易日盈余公告映射到下一市场交易日；收盘后公告需移到下一交易日，可提前标准化有效公告日。

In [ ]:
def check_extension_table(table, columns, keys):
    if not isinstance(table, pd.DataFrame):
        raise ValueError('输入必须是 DataFrame')
    missing = [name for name in columns if name not in table.columns]
    if missing:
        raise ValueError('缺字段: ' + ', '.join(missing))
    if table[keys].isna().any().any() or table.duplicated(keys).any():
        raise ValueError('主键缺失或重复: ' + ', '.join(keys))
    return table


def get_size_groups(caps, month_column='cap_month', value_column='market_value'):
    # 原股票月行保持不变，仅增加同月市值组别。
    check_extension_table(caps, ['Stkcd', month_column, value_column], ['Stkcd', month_column])
    if (caps[value_column].dropna() <= 0).any():
        raise ValueError('市值必须为正')
    caps['size_p30'] = caps.groupby(month_column)[value_column].transform(lambda x: x.quantile(0.30))
    caps['size_p70'] = caps.groupby(month_column)[value_column].transform(lambda x: x.quantile(0.70))
    caps['size_group'] = pd.Series(pd.NA, index=caps.index, dtype='string')
    valid = caps[value_column].notna()
    caps.loc[valid & (caps[value_column] <= caps['size_p30']), 'size_group'] = 'smallest'
    caps.loc[valid & (caps[value_column] > caps['size_p30']) & (caps[value_column] <= caps['size_p70']), 'size_group'] = 'middle'
    caps.loc[valid & (caps[value_column] > caps['size_p70']), 'size_group'] = 'largest'
    return caps


def add_extension_coverage(target, required, action):
    missing = [name for name in required if name not in extension_inputs]
    if missing:
        extension_coverage_rows.append({'target': target, 'status': '缺输入', 'detail': ', '.join(missing)})
        return
    try:
        output = action()
        extension_results[target] = output
        extension_coverage_rows.append({'target': target, 'status': '已计算，核查覆盖', 'detail': '实际输入结果；查看起止日、样本数与未匹配事件'})
    except (ValueError, KeyError, IndexError, np.linalg.LinAlgError) as error:
        extension_coverage_rows.append({'target': target, 'status': '输入/有效样本不足', 'detail': str(error)})

## Table 1：中国与美国盈余反应

每股补完整财季网格，delta=earnings(q)−earnings(q−4)，SUE=delta/连续八季delta的样本标准差（包含当季）。缺季不能压缩 shift(4)；σ=0则缺失。公告时市值组来自上一日历月。

CAR=窗口股票复利收益−同窗市场复利收益；k=0/3，市场交易日历上含首尾1/7日。股票缺任一市场日收益，则CAR缺失，不能用下一次股票成交替代停牌日。分组 pooled OLS 含截距、HC1，输出100*b、t与普通R²。CN样本2000–2016、US 1980–2016；实际覆盖不足时为局部检验，不能称完整论文复现。

In [ ]:
def get_earnings_surprises(earnings):
    check_extension_table(earnings, ['Stkcd', 'fiscal_quarter', 'announcement_date', 'earnings'], ['Stkcd', 'fiscal_quarter'])
    events = []
    for name, group in earnings.groupby('Stkcd', sort=False):
        quarterly = group.copy()
        quarterly['fiscal_quarter'] = pd.PeriodIndex(quarterly['fiscal_quarter'], freq='Q')
        if quarterly['fiscal_quarter'].duplicated().any():
            raise ValueError('转换后财季重复')
        quarterly = quarterly.set_index('fiscal_quarter').sort_index()
        grid = pd.period_range(quarterly.index.min(), quarterly.index.max(), freq='Q')
        quarterly = quarterly.reindex(grid)
        quarterly['announcement_date'] = pd.to_datetime(quarterly['announcement_date'])
        available = pd.to_datetime(quarterly['earnings_available_date']) if 'earnings_available_date' in quarterly else quarterly['announcement_date']
        delta_available = pd.concat([available,available.shift(4)],axis=1).max(axis=1)
        sigma_available = pd.to_datetime(delta_available.astype('int64').where(delta_available.notna()).rolling(8,min_periods=8).max())
        quarterly['delta'] = quarterly['earnings'] - quarterly['earnings'].shift(4)
        quarterly['delta_std'] = quarterly['delta'].rolling(8, min_periods=8).std()
        quarterly['sue'] = (quarterly['delta'] / quarterly['delta_std'].where(quarterly['delta_std'] > 0)).where(sigma_available.le(quarterly['announcement_date']))
        quarterly['Stkcd'] = name
        quarterly.index.name = 'fiscal_quarter'
        events.append(quarterly.dropna(subset=['announcement_date']).reset_index())
    if not events:
        raise ValueError('没有季度公告')
    return pd.concat(events, ignore_index=True)


def get_earnings_windows(earnings, daily, market, caps, k=0, after_close=False):
    check_extension_table(daily, ['Stkcd', 'Date', 'stock_return'], ['Stkcd', 'Date'])
    check_extension_table(market, ['Date', 'market_return'], ['Date'])
    if k not in [0, 3]:
        raise ValueError('Table1只使用k=0或3')
    events = get_earnings_surprises(earnings)
    events['announcement_date'] = pd.to_datetime(events['announcement_date'])
    events['cap_month'] = events['announcement_date'].dt.to_period('M') - 1
    monthly_caps = caps.copy()
    monthly_caps['cap_month'] = pd.PeriodIndex(monthly_caps['cap_month'], freq='M')
    monthly_caps = get_size_groups(monthly_caps)
    events = events.merge(monthly_caps[['Stkcd', 'cap_month', 'size_group']], on=['Stkcd', 'cap_month'], how='left', validate='many_to_one')
    market_days = market.copy()
    market_days['Date'] = pd.to_datetime(market_days['Date'])
    market_days = market_days.sort_values('Date').set_index('Date')
    prices = daily.copy()
    prices['Date'] = pd.to_datetime(prices['Date'])
    stock_daily = prices.set_index(['Stkcd', 'Date'])['stock_return']
    calendar = market_days.index
    abnormal = []
    event_dates = []
    for row in events.itertuples(index=False):
        position = calendar.searchsorted(row.announcement_date, side='right' if after_close else 'left')
        event_dates.append(calendar[position] if position < len(calendar) else pd.NaT)
        if position < k or position + k >= len(calendar):
            abnormal.append(np.nan)
            continue
        window = calendar[position-k:position+k+1]
        stock_index = pd.MultiIndex.from_product([[row.Stkcd], window], names=['Stkcd', 'Date'])
        stock_returns = stock_daily.reindex(stock_index)
        market_returns = market_days.loc[window, 'market_return']
        if stock_returns.isna().any() or market_returns.isna().any():
            abnormal.append(np.nan)
        else:
            abnormal.append((1 + stock_returns).prod() - (1 + market_returns).prod())
    events['event_trade_date'] = event_dates
    events['car'] = abnormal
    return events


def get_earnings_response(earnings, daily, market, caps, country, after_close=False):
    start = '2000-01-01' if country == 'China' else '1980-01-01'
    rows = []
    windows = {}
    for k in [0, 3]:
        events = get_earnings_windows(earnings, daily, market, caps, k, after_close)
        events = events[events['announcement_date'].between(start, '2016-12-31')].copy()
        windows[k] = events
        for group in ['smallest', 'middle', 'largest']:
            valid = events[events['size_group'] == group].replace([np.inf, -np.inf], np.nan).dropna(subset=['car', 'sue'])
            summary = {'country': country, 'panel': 'A' if k == 0 else 'B', 'size_group': group,
                       'n': len(valid), 'months': valid['announcement_date'].dt.to_period('M').nunique(), 'events_before_drop': int((events['size_group'] == group).sum()),
                       'unmatched_size_events': int(events['size_group'].isna().sum()),
                       'start': valid['announcement_date'].min(), 'end': valid['announcement_date'].max(),
                       'b_x100': np.nan, 't': np.nan, 'r2': np.nan}
            if len(valid) > 2 and valid['sue'].nunique() > 1:
                result = sm.OLS(valid['car'], sm.add_constant(valid[['sue']], has_constant='add')).fit(cov_type='HC1')
                summary.update({'b_x100': result.params['sue'] * 100, 't': result.tvalues['sue'], 'r2': result.rsquared})
            rows.append(summary)
    return {'table': pd.DataFrame(rows), 'events': windows}


for country, prefix in [('China', 'cn'), ('US', 'us')]:
    keys = [prefix + suffix for suffix in ['_earnings', '_daily', '_market', '_monthly_caps']]
    add_extension_coverage('table1_' + prefix, keys,
                           lambda prefix=prefix, country=country: get_earnings_response(
                               extension_inputs[prefix + '_earnings'], extension_inputs[prefix + '_daily'],
                               extension_inputs[prefix + '_market'], extension_inputs[prefix + '_monthly_caps'],
                               country, extension_options.get('earnings_release_after_close', False)))
for name in ['table1_cn', 'table1_us']:
    if name in extension_results:
        display(extension_results[name]['table'])

## Fig. 1：三个月前市值十分位

交易月前三个月末，全体上市股票市值断点用于归类借壳标的；deal_date不自动用董事会日或审批日代替。未匹配事件保留，缺失值不归小组；市值并列无法分十组时报告问题，不静默减少分组。当前输入事件数/匹配数与论文133另列，83%与超过一半只作为论文目标。

In [ ]:
def get_reverse_merger_sizes(deals, caps):
    check_extension_table(deals, ['deal_id', 'Stkcd', 'deal_date'], ['deal_id'])
    monthly_caps = caps.copy()
    monthly_caps['cap_month'] = pd.PeriodIndex(monthly_caps['cap_month'], freq='M')
    check_extension_table(monthly_caps, ['Stkcd', 'cap_month', 'market_value'], ['Stkcd', 'cap_month'])
    monthly_caps = monthly_caps.dropna(subset=['market_value'])
    if (monthly_caps['market_value'] <= 0).any():
        raise ValueError('市值必须为正')
    monthly_caps['size_decile'] = monthly_caps.groupby('cap_month')['market_value'].transform(
        lambda x: pd.qcut(x, q=10, labels=False) + 1)
    events = deals.copy()
    events['deal_date'] = pd.to_datetime(events['deal_date'])
    events = events[events['deal_date'].between('2007-01-01', '2016-12-31')].copy()
    events['cap_month'] = events['deal_date'].dt.to_period('M') - 3
    events = events.merge(monthly_caps[['Stkcd', 'cap_month', 'size_decile']], on=['Stkcd', 'cap_month'], how='left', validate='many_to_one')
    matched = events.dropna(subset=['size_decile'])
    distribution = matched['size_decile'].value_counts().reindex(range(1, 11), fill_value=0).rename('events').to_frame()
    distribution.index.name = 'size_decile'
    distribution['fraction'] = distribution['events'] / len(matched) if len(matched) else np.nan
    coverage = pd.DataFrame([{'events': len(events), 'matched': len(matched), 'missing_caps': len(events)-len(matched),
                             'paper_events': 133, 'bottom30_fraction': matched['size_decile'].le(3).mean(),
                             'bottom10_fraction': matched['size_decile'].eq(1).mean()}])
    return {'events': events, 'distribution': distribution, 'coverage': coverage}


def plot_reverse_merger_sizes(output):
    if output['distribution']['fraction'].notna().any():
        ax = output['distribution']['fraction'].plot.bar(color='#285e8d', figsize=(8, 4))
        ax.set(xlabel='Size decile (three months before deal)', ylabel='Fraction of matched events',
               title='Reverse-merger shells: supplied sample')
        plt.tight_layout()
        plt.show()
    display(output['coverage'])


add_extension_coverage('fig1', ['reverse_mergers', 'cn_monthly_caps'],
                       lambda: get_reverse_merger_sizes(extension_inputs['reverse_mergers'], extension_inputs['cn_monthly_caps']))
if 'fig1' in extension_results:
    plot_reverse_merger_sizes(extension_results['fig1'])

## Fig. 2：730自然日滚动壳价值

Fig.1先用借壳交易前三个月前的市值十分位识别小壳集中度：论文约83%的133起事件位于最小30%组；因此CH-3按股票市值剔除最小30%。Fig.2再用事件窗口前后估值变化估计壳价值比例，报告的平均值约29.5%。所以“30%”是依据壳价值证据设定的股票池截断点，不是直接把每家公司30%的市值定义成壳价值。Appendix A4：在[t−729,t]这个730日闭区间，p=发生借壳的最小30%股票比例/2（论文近似减半），同股重复交易计一次，分母为t日完整截面最小30%股票数。G=事件窗口已结束且终点落在该区间内的总市值变化均值，S=pG/(0.03+p)，V=当日最小30%市值中位数。图画S/V与S（人民币百万元）。

开始端点是董事会−60日、结束端点审批+60日。自然日端点用截至该日最后报价，并报告所用日期和stale_days；交易日基准用全市场交易日历移动60日。未到未来端点，不能用最后一笔历史报价冒充完成窗口。增发对总市值的影响按论文保留，不能替换成股价收益。
仅在完整730日事件历史、确知没有事件时p=0且S=0；需要G但没有完成窗口时保留缺失。A4没有进一步说明事件最小组重分类时点，本实现明确使用Fig.1标记；原始WIND规则仍待数据核查。

In [ ]:
def get_reverse_merger_windows(deals, daily, day_basis='calendar'):
    check_extension_table(deals, ['deal_id', 'Stkcd', 'board_date', 'approval_date'], ['deal_id'])
    check_extension_table(daily, ['Stkcd', 'Date', 'market_value', 'stock_return'], ['Stkcd', 'Date'])
    events = deals.copy()
    for column in ['board_date', 'approval_date']:
        events[column] = pd.to_datetime(events[column])
    if events[['board_date', 'approval_date']].isna().any().any():
        raise ValueError('缺董事会或审批日')
    if (events['approval_date'] < events['board_date']).any():
        raise ValueError('审批日早于提案日')
    quotes = daily.copy()
    quotes['Date'] = pd.to_datetime(quotes['Date'])
    calendar = pd.DatetimeIndex(quotes['Date'].drop_duplicates().sort_values())
    if day_basis == 'calendar':
        events['window_start'] = events['board_date'] - pd.Timedelta(days=60)
        events['window_end'] = events['approval_date'] + pd.Timedelta(days=60)
    elif day_basis == 'trading':
        starts = []
        ends = []
        for row in events.itertuples(index=False):
            board_pos = calendar.searchsorted(row.board_date, side='left')
            approval_pos = calendar.searchsorted(row.approval_date, side='left')
            starts.append(calendar[board_pos-60] if board_pos >= 60 and board_pos < len(calendar) else pd.NaT)
            ends.append(calendar[approval_pos+60] if approval_pos+60 < len(calendar) else pd.NaT)
        events['window_start'] = starts
        events['window_end'] = ends
    else:
        raise ValueError('day_basis应为calendar或trading')
    for endpoint in ['start', 'end']:
        values = []
        quote_dates = []
        for row in events.itertuples(index=False):
            limit = getattr(row, 'window_' + endpoint)
            history = quotes[(quotes['Stkcd'] == row.Stkcd) & (quotes['Date'] <= limit)].sort_values('Date')
            last = history.iloc[-1] if len(history) else None
            values.append(last['market_value'] if last is not None else np.nan)
            quote_dates.append(last['Date'] if last is not None else pd.NaT)
        events[endpoint + '_market_value'] = values
        events[endpoint + '_quote_date'] = quote_dates
        events[endpoint + '_stale_days'] = (events['window_' + endpoint] - events[endpoint + '_quote_date']).dt.days
    events['value_gain'] = events['end_market_value'] - events['start_market_value']
    events.loc[events['window_end'] > calendar.max(), 'value_gain'] = np.nan
    return events


def get_shell_values(deals, daily, caps, history_start='2007-01-01', day_basis='calendar', discount_rate=0.03):
    size_output = get_reverse_merger_sizes(deals, caps)
    if size_output['events']['size_decile'].isna().any():
        raise ValueError('借壳事件三个月前市值未完全匹配')
    events = get_reverse_merger_windows(size_output['events'], daily, day_basis)
    events = events[events['size_decile'] <= 3].copy()
    quotes = daily.copy()
    quotes['Date'] = pd.to_datetime(quotes['Date'])
    quotes = quotes.dropna(subset=['market_value'])
    if (quotes['market_value'] <= 0).any() or discount_rate <= 0:
        raise ValueError('市值和贴现率必须为正')
    rows = []
    history_start = pd.Timestamp(history_start)
    for day, group in quotes.groupby('Date', sort=True):
        if not pd.Timestamp('2009-01-01') <= day <= pd.Timestamp('2016-12-31'):
            continue
        bottom = group[group['market_value'] <= group['market_value'].quantile(0.30)]
        cutoff = day - pd.Timedelta(days=729)
        past = events[events['deal_date'].between(cutoff, day)]
        completed = events[events['window_end'].between(cutoff, day)]
        p = past['Stkcd'].nunique() / len(bottom) / 2 if len(bottom) and cutoff >= history_start else np.nan
        g = completed['value_gain'].mean() if completed['value_gain'].notna().all() else np.nan
        shell_value = 0.0 if p == 0 else p * g / (discount_rate + p)
        median_cap = bottom['market_value'].median()
        rows.append({'Date': day, 'p': p, 'g': g, 'shell_value': shell_value, 'median_bottom30_cap': median_cap,
                     'shell_to_market': shell_value / median_cap, 'shell_millions': shell_value / 1e6,
                     'bottom30_stocks': len(bottom), 'shell_stocks_730d': past['Stkcd'].nunique(),
                     'completed_events_730d': len(completed), 'valid_g_events': completed['value_gain'].notna().sum()})
    if not rows:
        raise ValueError('没有2009–2016日度截面')
    values = pd.DataFrame(rows).set_index('Date')
    summary = pd.DataFrame([{'start': values.index.min(), 'end': values.index.max(), 'days': len(values),
                             'valid_ratio_days': values['shell_to_market'].notna().sum(),
                             'mean_shell_to_market': values['shell_to_market'].mean(), 'paper_mean_ratio': 0.295,
                             'day_basis': day_basis, 'discount_rate': discount_rate}])
    return {'daily': values, 'events': events, 'summary': summary}


def plot_shell_values(output):
    series = output['daily']
    if series['shell_value'].notna().any():
        fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
        axes[0].plot(series.index, series['shell_to_market'], color='#285e8d')
        axes[0].set(ylabel='S / V', title='Estimated shell value: supplied sample')
        axes[1].plot(series.index, series['shell_millions'], color='#9b5540')
        axes[1].set(ylabel='Shell value (million RMB)', xlabel='Date')
        fig.tight_layout()
        plt.show()
    display(output['summary'])


add_extension_coverage('fig2', ['reverse_mergers', 'cn_shell_daily', 'cn_monthly_caps'],
                       lambda: get_shell_values(extension_inputs['reverse_mergers'], extension_inputs['cn_shell_daily'],
                                                extension_inputs['cn_monthly_caps'], extension_options.get('reverse_history_start', '2007-01-01'),
                                                extension_options.get('event_day_basis', 'calendar')))
if 'fig2' in extension_results:
    plot_shell_values(extension_results['fig2'])

## Table A2：壳价值代理与收益

上一月完整股票池总市值30/70分位形成三组，各组value weighted月收益。对每组OLS：R_group=a+b*proxy+gamma*other_two_groups+error，HC1。

A面板：每一天所有处于董事会−60日至审批+60日窗口的借壳股票平均收益，然后按月复利；同股窗口重叠不重复计。没有活动事件的日子不能擅自填零，当前口径将有缺失日的月份标NaN并报告覆盖。论文没明确此类缺失处理，须披露对比限制。
B面板：真实证券主表当月IPO个数log(NIPO)，零IPO月为NaN、报告并排除；论文没有交代零IPO月处理，不静默改log(1+NIPO)。小组论文log IPO系数−0.023、t=−1.53，不能称显著负相关。

In [ ]:
def get_size_group_returns(monthly):
    check_extension_table(monthly, ['Stkcd', 'month', 'lagged_market_value', 'stock_return'], ['Stkcd', 'month'])
    panel = monthly.copy()
    panel['month'] = pd.PeriodIndex(panel['month'], freq='M')
    panel = get_size_groups(panel, 'month', 'lagged_market_value')
    panel = panel.dropna(subset=['lagged_market_value', 'stock_return', 'size_group'])
    panel['weighted_return'] = panel['lagged_market_value'] * panel['stock_return']
    sums = panel.groupby(['month', 'size_group'])[['weighted_return', 'lagged_market_value']].sum()
    sums['group_return'] = sums['weighted_return'] / sums['lagged_market_value']
    return sums['group_return'].unstack('size_group').reindex(columns=['smallest', 'middle', 'largest'])


def get_reverse_merger_returns(deals, daily, day_basis='calendar'):
    events = get_reverse_merger_windows(deals, daily, day_basis)
    quotes = daily[['Stkcd', 'Date', 'stock_return']].copy()
    quotes['Date'] = pd.to_datetime(quotes['Date'])
    active = []
    for row in events.itertuples(index=False):
        selected = quotes[(quotes['Stkcd'] == row.Stkcd) & quotes['Date'].between(row.window_start, row.window_end)]
        active.append(selected)
    event_quotes = pd.concat(active, ignore_index=True).drop_duplicates(['Stkcd', 'Date']) if active else quotes.iloc[:0]
    daily_returns = event_quotes.groupby('Date')['stock_return'].mean().rename('rm_daily')
    calendar = pd.DatetimeIndex(quotes['Date'].drop_duplicates().sort_values())
    panel = daily_returns.reindex(calendar).to_frame()
    panel['month'] = panel.index.to_period('M')
    rows = []
    for month, group in panel.groupby('month'):
        complete = group['rm_daily'].notna().all()
        rows.append({'month': month, 'rm': (1 + group['rm_daily']).prod() - 1 if complete else np.nan,
                     'event_days': group['rm_daily'].notna().sum(), 'market_days': len(group)})
    if not rows:
        raise ValueError('缺交易日历')
    return pd.DataFrame(rows).set_index('month')


def get_ipo_proxy(master, months):
    check_extension_table(master, ['Stkcd', 'listing_date'], ['Stkcd'])
    master = master.copy()
    master['listing_date'] = pd.to_datetime(master['listing_date'])
    if master['listing_date'].isna().any():
        raise ValueError('真实上市日缺失')
    counts = master.groupby(master['listing_date'].dt.to_period('M')).size().reindex(months, fill_value=0)
    return pd.DataFrame({'nipo': counts, 'log_nipo': np.log(counts.where(counts > 0))})


def get_shell_return_regressions(group_returns, proxy, proxy_name):
    if not group_returns.index.is_unique or not proxy.index.is_unique:
        raise ValueError('月份主键重复')
    aligned = group_returns.join(proxy.rename('shell_proxy'), how='inner')
    aligned = aligned.loc[pd.Period('2007-01', freq='M'):pd.Period('2016-12', freq='M')]
    valid = aligned.replace([np.inf, -np.inf], np.nan).dropna()
    if len(valid) <= 4:
        raise ValueError('TableA2有效月份不足五个月')
    rows = []
    for name in ['smallest', 'middle', 'largest']:
        controls = [column for column in ['smallest', 'middle', 'largest'] if column != name]
        design = sm.add_constant(valid[['shell_proxy'] + controls], has_constant='add')
        if np.linalg.matrix_rank(design) < design.shape[1]:
            raise ValueError('TableA2解释变量线性相关')
        result = sm.OLS(valid[name], design).fit(cov_type='HC1')
        rows.append({'proxy': proxy_name, 'size_group': name, 'b': result.params['shell_proxy'],
                     't': result.tvalues['shell_proxy'], 'r2': result.rsquared, 'months': len(valid),
                     'dropped_months': len(aligned)-len(valid), 'start': valid.index.min(), 'end': valid.index.max()})
    return pd.DataFrame(rows)


def get_table_a2_rm(monthly, deals, daily, day_basis='calendar'):
    groups = get_size_group_returns(monthly)
    proxy = get_reverse_merger_returns(deals, daily, day_basis)
    return {'table': get_shell_return_regressions(groups, proxy['rm'], 'rm'), 'proxy_coverage': proxy}


def get_table_a2_ipo(monthly, master):
    groups = get_size_group_returns(monthly)
    proxy = get_ipo_proxy(master, groups.index)
    return {'table': get_shell_return_regressions(groups, proxy['log_nipo'], 'log_nipo'), 'proxy_coverage': proxy}


add_extension_coverage('table_a2_rm', ['cn_monthly_full', 'reverse_mergers', 'cn_shell_daily'],
                       lambda: get_table_a2_rm(extension_inputs['cn_monthly_full'], extension_inputs['reverse_mergers'],
                                               extension_inputs['cn_shell_daily'], extension_options.get('event_day_basis', 'calendar')))
add_extension_coverage('table_a2_ipo', ['cn_monthly_full', 'cn_ipo'],
                       lambda: get_table_a2_ipo(extension_inputs['cn_monthly_full'], extension_inputs['cn_ipo']))
for name in ['table_a2_rm', 'table_a2_ipo']:
    if name in extension_results:
        display(extension_results[name]['table'])
        display(extension_results[name]['proxy_coverage'])

## Fig. A1：贝叶斯核验边界

A7使用等模型先验，横轴sqrt(E_prior[S_max²])/S_market，样本2000–2016共204月。来源：[Barillas & Shanken (2018)](https://onlinelibrary.wiley.com/doi/10.1111/jofi.12607)、[Stambaugh & Yuan (2017)](https://academic.oup.com/rfs/article/30/4/1270/2965095)。原始解析推导PDF本次访问403，未逐项核验先验常数/模型维度积分，默认不能从因子数值复现Fig.A1。

给出已核验bayesian_log_ml与bayesian_provenance后，下列函数仅完成论文式(4)的Bayes归一化并绘图，明确外部边际似然；没有声称实现式(5)解析积分。共同测试资产在比较中相消，GRS p、逆GRS、Sharpe排名都不能替代边际似然。

更新：`07_bayesian_comparison.ipynb` 已找到原始Barillas–Shanken最终稿并实现解析边际似然；主入口在本文件之后运行07。这里保留外部logML接口，旧“未核验”状态会由07的真实计算覆盖。

In [ ]:
def get_model_posteriors(log_ml, provenance):
    check_extension_table(log_ml, ['multiplier', 'ch3_log_ml', 'ff3_log_ml'], ['multiplier'])
    if not isinstance(provenance, str) or not provenance.strip():
        raise ValueError('需提供已核验边际似然来源及实现记录')
    posterior = log_ml.sort_values('multiplier').copy()
    likelihoods = posterior[['ch3_log_ml', 'ff3_log_ml']].to_numpy(dtype=float)
    if not np.isfinite(likelihoods).all():
        raise ValueError('对数边际似然必须有限')
    likelihoods = np.exp(likelihoods - likelihoods.max(axis=1, keepdims=True))
    probabilities = likelihoods / likelihoods.sum(axis=1, keepdims=True)
    posterior['ch3_probability'] = probabilities[:, 0]
    posterior['ff3_probability'] = probabilities[:, 1]
    posterior.attrs['provenance'] = provenance
    return posterior


if 'bayesian_log_ml' in extension_inputs and extension_options.get('bayesian_provenance'):
    add_extension_coverage('fig_a1_external_ml', ['bayesian_log_ml'],
                           lambda: get_model_posteriors(extension_inputs['bayesian_log_ml'], extension_options['bayesian_provenance']))
    if 'fig_a1_external_ml' in extension_results:
        posterior = extension_results['fig_a1_external_ml']
        ax = posterior.plot(x='multiplier', y=['ch3_probability', 'ff3_probability'], figsize=(8, 4))
        ax.set(xlabel='Prior Sharpe ratio multiplier', ylabel='Posterior probability',
               ylim=(0, 1), title='Bayes equation (4): externally verified marginal likelihoods')
        plt.tight_layout()
        plt.show()
else:
    extension_coverage_rows.append({'target': 'fig_a1', 'status': '解析边际似然未核验',
                                    'detail': '需Barillas–Shanken完整积分与先验校准，不用GRS/Sharpe排名替代'})

## 壳剔除断点：0%、25%、30%、35%

可直接消费主notebook的monthly_all与get_ch3_factors。各断点重新生成因子，再在相同完整月份比较均值百分比、White HC1均值t及相对30%因子的相关性。0%包含壳股票；25/35%对应论文稳健性界限。相关高不等于定价能力相同；GRS/异象alpha需要固定共同测试资产，此处仅因子诊断，不编造论文未列的断点数值。

In [ ]:
def get_shell_cut_sensitivity(monthly, factor_function):
    factors = {}
    columns = ['market_factor', 'size_factor_smb', 'value_factor_vmg']
    for cut in [0.0, 0.25, 0.30, 0.35]:
        factors[cut] = factor_function(monthly.copy(), cut=cut)
        if not isinstance(factors[cut], pd.DataFrame):
            raise ValueError('get_ch3_factors应返回DataFrame')
        if not factors[cut].index.is_unique:
            raise ValueError('因子月份重复')
    common = factors[0.30].dropna(subset=columns).index
    for matrix in factors.values():
        common = common.intersection(matrix.dropna(subset=columns).index)
    common = common.sort_values()
    if len(common) < 3:
        raise ValueError('共同完整月份不足')
    rows = []
    for cut, matrix in factors.items():
        for column in columns:
            series = matrix.loc[common, column]
            result = sm.OLS(series.to_numpy(), np.ones((len(series), 1))).fit(cov_type='HC1')
            rows.append({'cut': cut, 'factor': column, 'mean_percent': series.mean() * 100,
                         'mean_t_hc1': result.tvalues[0], 'corr_cut30': series.corr(factors[0.30].loc[common, column]),
                         'months': len(common), 'start': common.min(), 'end': common.max()})
    return {'summary': pd.DataFrame(rows), 'factors': factors, 'common_months': common}


if 'monthly_all' in globals() and 'get_ch3_factors' in globals():
    try:
        extension_results['shell_cut_sensitivity'] = get_shell_cut_sensitivity(monthly_all, get_ch3_factors)
        display(extension_results['shell_cut_sensitivity']['summary'])
        extension_coverage_rows.append({'target': 'shell_cut_sensitivity', 'status': '已计算', 'detail': '四断点相同月份，因子诊断'})
    except (ValueError, KeyError, TypeError) as error:
        extension_coverage_rows.append({'target': 'shell_cut_sensitivity', 'status': '输入契约不足', 'detail': str(error)})
else:
    extension_coverage_rows.append({'target': 'shell_cut_sensitivity', 'status': '缺主notebook输出', 'detail': '需monthly_all与get_ch3_factors'})

## Table A3：真实历史金融与房地产分类

新增可选输入 `extension_inputs['cn_financial_classification']`：一行股票×形成月，主键 `(Stkcd, formation_date)`，列 `is_financial,is_real_estate` 为非缺失布尔值，历史有效分类不可用当前分类回填过去。formation_date 为 Period[M]。

论文 A6 同时排除房地产、银行和金融服务（平均79家房地产、20家其他金融）。与主 notebook 的 `characteristic_panel` 按股票及形成月严格一对一合并，完全覆盖后才筛除金融/房地产，使用同一个 `get_characteristic_fm` 重跑九规格、NW(4)。没有历史分类时阻塞；不能把粗行业编码或股票名字猜作非金融。

In [ ]:
def get_nonfinancial_characteristics(panel, classification):
    check_extension_table(classification, ['Stkcd', 'formation_date', 'is_financial', 'is_real_estate'], ['Stkcd', 'formation_date'])
    labels = classification.copy()
    labels['formation_date'] = pd.PeriodIndex(labels['formation_date'], freq='M')
    check_extension_table(labels, list(labels.columns), ['Stkcd', 'formation_date'])
    for column in ['is_financial', 'is_real_estate']:
        if not pd.api.types.is_bool_dtype(labels[column]) or labels[column].isna().any():
            raise ValueError('历史分类必须是非缺失布尔值: ' + column)
    data = panel.copy()
    data['formation_date'] = pd.PeriodIndex(data['formation_date'], freq='M')
    check_extension_table(data, ['Stkcd', 'formation_date'], ['Stkcd', 'formation_date'])
    data = data.merge(labels[['Stkcd', 'formation_date', 'is_financial', 'is_real_estate']], on=['Stkcd', 'formation_date'], how='left', validate='one_to_one')
    if data[['is_financial', 'is_real_estate']].isna().any().any():
        raise ValueError('历史金融/房地产分类未覆盖全部特征股票月')
    exclusion = data['is_financial'] | data['is_real_estate']
    coverage = data.assign(excluded=exclusion).groupby('formation_date').agg(stocks=('Stkcd', 'size'), excluded=('excluded', 'sum'))
    return data.loc[~exclusion].drop(columns=['is_financial', 'is_real_estate']), coverage


if 'cn_financial_classification' in extension_inputs and 'characteristic_panel' in globals() and 'get_characteristic_fm' in globals():
    try:
        nonfinancial_panel, financial_exclusion_coverage = get_nonfinancial_characteristics(characteristic_panel, extension_inputs['cn_financial_classification'])
        table_a3_result, table_a3_slopes = get_characteristic_fm(nonfinancial_panel)
        if table_a3_result.empty:
            raise ValueError('非金融特征回归没有有效月份')
        extension_results['table_a3'] = {'table': table_a3_result, 'slopes': table_a3_slopes, 'exclusions': financial_exclusion_coverage}
        if 'record_result' in globals():
            for row in table_a3_result.to_dict('records'):
                for metric in ['estimate', 't']:
                    record_result('a3', row['panel'], row['row'], metric, row[metric], row['months'], status='真实历史分类排除金融及房地产')
        display(table_a3_result)
        display(financial_exclusion_coverage)
        extension_coverage_rows.append({'target': 'table_a3', 'status': '已计算', 'detail': '历史分类全覆盖；同九规格与NW(4)'})
    except (ValueError, KeyError) as error:
        extension_coverage_rows.append({'target': 'table_a3', 'status': '输入/样本不足', 'detail': str(error)})
else:
    extension_coverage_rows.append({'target': 'table_a3', 'status': '缺历史金融分类或主回归输入', 'detail': '需cn_financial_classification、characteristic_panel、get_characteristic_fm'})

## 37行业的解释力诊断

新增可选输入 `cn_industry37_classification`：一行股票×形成月，主键 `(Stkcd, formation_date)`，列 `industry37` 是历史有效申万37行业标识。不得用现在申万分类替代旧37行业体系。与主 `monthly_final` 前月市值和月超额收益对齐，形成行业value weighted超额收益。行业和CH3/市场在**相同完整月份**回归、逐行业普通R²，再做行业间等权平均。

论文正文报告CH3约82%、市场约72%。这些是目标，不是本地结果。若有效行业数不是37，诊断明确标记实际行业数，不能把均值称为原37行业复现。

In [ ]:
def get_industry37_returns(panel, classification):
    check_extension_table(classification, ['Stkcd', 'formation_date', 'industry37'], ['Stkcd', 'formation_date'])
    labels = classification.copy()
    labels['formation_date'] = pd.PeriodIndex(labels['formation_date'], freq='M')
    check_extension_table(labels, ['Stkcd', 'formation_date', 'industry37'], ['Stkcd', 'formation_date'])
    data = panel.copy()
    data['formation_date'] = pd.PeriodIndex(data['formation_date'], freq='M')
    check_extension_table(data, ['Stkcd', 'formation_date', 'yield_date', 'monthly_market_value', 'monthly_excess_return'], ['Stkcd', 'formation_date'])
    data = data.merge(labels[['Stkcd', 'formation_date', 'industry37']], on=['Stkcd', 'formation_date'], how='left', validate='one_to_one')
    if data['industry37'].isna().any():
        raise ValueError('历史申万37行业未覆盖全部股票月')
    data = data.dropna(subset=['monthly_market_value', 'monthly_excess_return'])
    data['weighted_return'] = data['monthly_market_value'] * data['monthly_excess_return']
    totals = data.groupby(['yield_date', 'industry37'])[['weighted_return', 'monthly_market_value']].sum()
    return (totals['weighted_return'] / totals['monthly_market_value']).unstack('industry37')


def get_industry37_explanatory_power(returns, factors):
    common = returns.join(factors, how='inner').replace([np.inf, -np.inf], np.nan).dropna()
    if len(common) <= factors.shape[1] + 2:
        raise ValueError('37行业共同完整月份不足')
    rows = []
    for industry in returns.columns:
        for name, model in [('ch3', factors), ('capm', factors[['market_factor']])]:
            design = sm.add_constant(common[model.columns], has_constant='add')
            if np.linalg.matrix_rank(design) < design.shape[1]:
                raise ValueError('行业因子线性相关')
            result = sm.OLS(common[industry], design).fit(cov_type='HC1')
            rows.append({'industry37': industry, 'model': name, 'r2': result.rsquared, 'months': len(common)})
    result = pd.DataFrame(rows)
    summary = result.groupby('model').agg(mean_r2=('r2', 'mean'), industries=('industry37', 'nunique'), months=('months', 'first'))
    return {'table': result, 'summary': summary, 'common_months': common.index}


if 'cn_industry37_classification' in extension_inputs and 'monthly_final' in globals() and 'f_ch3' in globals():
    try:
        industry37_returns = get_industry37_returns(monthly_final, extension_inputs['cn_industry37_classification'])
        extension_results['industry37'] = get_industry37_explanatory_power(industry37_returns, f_ch3)
        display(extension_results['industry37']['summary'])
        extension_coverage_rows.append({'target': 'industry37', 'status': '已计算，核查行业数', 'detail': '历史申万37；相同完整月份'})
    except (ValueError, KeyError) as error:
        extension_coverage_rows.append({'target': 'industry37', 'status': '输入/样本不足', 'detail': str(error)})
else:
    extension_coverage_rows.append({'target': 'industry37', 'status': '缺历史行业分类', 'detail': '需cn_industry37_classification与主因子面板'})

## 反转1/3/6/12/60月及2012中价检验

原文未列表的诊断：过去1、3、6、12月及5年收益对未来反转预测依次变弱，1月CAPM年化alpha约18%、12月约6%。形成期到持有期要错开，不能用持有月收益生成自己的信号。每股完整月历，过去h个连续自然月复利（含刚结束的形成月，无skip），缺月信号缺失；低历史收益组做多，高组做空，value weighted十组。

新增可选 `cn_reversal_monthly`：股票×持有月，主键Stkcd,month，列stock_return,excess_return,lagged_market_value,eligible；eligible为形成时可进入论文剔除壳股票池的非缺失布尔值。完整历史计算信号，eligible只用于持有组合筛选。未提供时使用主monthly未筛选月收益历史，按完整自然月历计算，再对接monthly_all的滞后市值与monthly_final的形成时资格。不能用已筛选的monthly_all构建回望历史。60月窗口在数据开始5年后才有效，不能拿204月样本声称全部覆盖。

论文用2012–2016最佳买卖报价中价验证一月反转，月均多空2.21%对收盘价2.15%。现有数据没有bid/ask，以下明确保留未完成；收盘价不能冒充中价。

In [ ]:
def get_reversal_horizon_returns(monthly, horizons=(1, 3, 6, 12, 60)):
    check_extension_table(monthly, ['Stkcd', 'month', 'stock_return', 'excess_return', 'lagged_market_value', 'eligible'], ['Stkcd', 'month'])
    data = monthly.copy()
    data['month'] = pd.PeriodIndex(data['month'], freq='M')
    if not pd.api.types.is_bool_dtype(data['eligible']) or data['eligible'].isna().any():
        raise ValueError('eligible必须为非缺失布尔值')
    histories = data.pivot(index='month', columns='Stkcd', values='stock_return').sort_index()
    histories = histories.reindex(pd.period_range(histories.index.min(), histories.index.max(), freq='M'))
    spreads = {}
    counts = []
    for horizon in horizons:
        signals = (1 + histories).rolling(horizon, min_periods=horizon).apply(np.prod, raw=True).shift(1) - 1
        signals = signals.rename_axis(index='month', columns='Stkcd').stack().rename('past_return').reset_index()
        selected = data.merge(signals, on=['Stkcd', 'month'], how='left', validate='one_to_one')
        selected = selected[selected['eligible']].dropna(subset=['past_return', 'excess_return', 'lagged_market_value'])
        selected = selected[selected['lagged_market_value'] > 0]
        selected['reversal_decile'] = selected.groupby('month')['past_return'].transform(
            lambda x: pd.qcut(x.rank(method='first'), q=10, labels=False)+1 if len(x) >= 10 else np.repeat(np.nan, len(x)))
        selected['weighted_return'] = selected['lagged_market_value'] * selected['excess_return']
        legs = selected.dropna(subset=['reversal_decile']).groupby(['month', 'reversal_decile'])[['weighted_return', 'lagged_market_value']].sum()
        returns = (legs['weighted_return'] / legs['lagged_market_value']).unstack('reversal_decile').reindex(columns=range(1, 11))
        spreads[horizon] = returns[1] - returns[10]
        counts.append({'horizon': horizon, 'stock_months': len(selected), 'months': spreads[horizon].notna().sum(),
                       'start': spreads[horizon].dropna().index.min(), 'end': spreads[horizon].dropna().index.max()})
    return {'returns': pd.DataFrame(spreads), 'coverage': pd.DataFrame(counts)}


if 'cn_reversal_monthly' in extension_inputs or all(name in globals() for name in ['monthly', 'monthly_all', 'monthly_final']):
    try:
        if 'cn_reversal_monthly' in extension_inputs:
            reversal_monthly = extension_inputs['cn_reversal_monthly']
            reversal_status = '完整月历史，eligible仅筛组合'
        else:
            reversal_monthly = monthly[['Stkcd', 'yield_date', 'monthly_return', 'monthly_excess_return']].copy()
            reversal_monthly = reversal_monthly.merge(monthly_all[['Stkcd', 'yield_date', 'monthly_market_value']], on=['Stkcd', 'yield_date'], how='left', validate='one_to_one')
            eligible_keys = monthly_final[['Stkcd', 'yield_date']].copy()
            eligible_keys['eligible'] = True
            reversal_monthly = reversal_monthly.merge(eligible_keys, on=['Stkcd', 'yield_date'], how='left', validate='one_to_one')
            reversal_monthly['eligible'] = reversal_monthly['eligible'].eq(True)
            reversal_monthly = reversal_monthly.rename(columns={'yield_date': 'month', 'monthly_return': 'stock_return', 'monthly_excess_return': 'excess_return', 'monthly_market_value': 'lagged_market_value'})
            reversal_status = '未筛选monthly完整历史；monthly_final仅确定组合资格；60月起始2005'
        reversal_output = get_reversal_horizon_returns(reversal_monthly)
        rows = []
        if 'run_ts' in globals() and 'f_ch3' in globals():
            for horizon in reversal_output['returns']:
                result = run_ts(reversal_output['returns'][horizon], f_ch3[['market_factor']])
                if result is not None:
                    rows.append({'horizon': horizon, 'annual_alpha_percent': 1200 * result.params['const'],
                                 'alpha_t_hc1': result.tvalues['const'], 'months': int(result.nobs)})
        reversal_output['capm'] = pd.DataFrame(rows)
        extension_results['reversal_horizons'] = reversal_output
        display(reversal_output['coverage'])
        display(reversal_output['capm'])
        extension_coverage_rows.append({'target': 'reversal_horizons', 'status': '已计算，核查历史覆盖', 'detail': reversal_status})
    except (ValueError, KeyError, IndexError) as error:
        extension_coverage_rows.append({'target': 'reversal_horizons', 'status': '输入/窗口不足', 'detail': str(error)})
else:
    extension_coverage_rows.append({'target': 'reversal_horizons', 'status': '缺月历史', 'detail': '需cn_reversal_monthly或monthly/monthly_all/monthly_final'})
extension_coverage_rows.append({'target': 'midquote_reversal_2012_2016', 'status': '缺最佳bid/ask',
                                'detail': '收盘价不能替代中价；原文2.21%/2.15%只是对照目标'})

# 把真正得到的Table1/A2接入主表比较；缺失结果不会填论文原值。
if 'record_result' in globals():
    for key, country in [('table1_cn', 'china'), ('table1_us', 'us')]:
        if key in extension_results:
            for row in extension_results[key]['table'].to_dict('records'):
                panel = ('k0_' if row['panel'] == 'A' else 'k3_') + country
                for output_name, metric in [('b_x100', 'beta'), ('t', 't_beta'), ('r2', 'r2')]:
                    record_result('table1', panel, row['size_group'], metric, row[output_name], row['months'], status='实际公告事件检验，查看覆盖')
    for key, panel in [('table_a2_rm', 'rm'), ('table_a2_ipo', 'log_nipo')]:
        if key in extension_results:
            for row in extension_results[key]['table'].to_dict('records'):
                for output_name, metric in [('b', 'estimate'), ('t', 't')]:
                    record_result('a2', panel, row['size_group'], metric, row[output_name], row['months'], status='真实输入壳代理，缺失处理已披露')

## 覆盖与结论边界

extension_coverage按项目报告状态；extension_results只含真正得到的结果。检查实际起止日、n/月份数、未匹配分组、完成窗口和stale_days、事件日覆盖、零IPO月份。TableA2论文RM系数为0.963/−1.267/−0.330；log IPO为−0.023/0.013/−0.001，但不能当本地估计。A6非金融马赛跑和A7因子互解释由主结果notebook处理。

In [ ]:
extension_coverage = pd.DataFrame(extension_coverage_rows)
display(extension_coverage)

## 微型行为检查：人工小表不作实证结论

只验证连续季度和缺季、公告交易日历和前月分组、事件窗口未完成禁止G、730自然日边界、公式与等先验归一化；这些表不进入extension_results。

In [ ]:
def check_extended_analysis_behaviour():
    checks = []
    quarters = pd.period_range('2000Q1', periods=16, freq='Q')
    earnings = pd.DataFrame({'Stkcd': 'a', 'fiscal_quarter': quarters,
                             'announcement_date': quarters.to_timestamp(how='end').normalize() + pd.Timedelta(days=30),
                             'earnings': np.arange(16, dtype=float) ** 2})
    sue = get_earnings_surprises(earnings)
    assert sue['sue'].notna().sum() == 5
    gap = get_earnings_surprises(earnings.drop(index=4))
    assert pd.isna(gap.loc[gap['fiscal_quarter'] == pd.Period('2002Q1'), 'delta'].iloc[0])
    checks.append({'check': '财季网格和缺季', 'passed': True})

    days = pd.bdate_range('2000-01-01', '2004-06-30')
    daily = pd.DataFrame({'Stkcd': 'a', 'Date': days, 'stock_return': 0.01})
    market = pd.DataFrame({'Date': days, 'market_return': 0.005})
    caps = pd.DataFrame([{'Stkcd': name, 'cap_month': month, 'market_value': value}
                         for month in pd.period_range('2000-01', '2004-06', freq='M')
                         for name, value in [('a', 1.0), ('b', 2.0), ('c', 3.0)]])
    windows = get_earnings_windows(earnings, daily, market, caps, k=3)
    usable = windows.dropna(subset=['sue', 'car'])
    assert np.allclose(usable['car'], 1.01**7 - 1.005**7)
    assert usable['size_group'].eq('smallest').all()
    removed_day = usable['event_trade_date'].iloc[0]
    missing = get_earnings_windows(earnings, daily[daily['Date'] != removed_day], market, caps, k=3)
    assert pd.isna(missing.loc[missing['event_trade_date'] == removed_day, 'car'].iloc[0])
    checks.append({'check': '七日CAR/前月分组/缺日', 'passed': True})

    shell_days = pd.bdate_range('2007-01-01', '2009-04-30')
    shell_daily = pd.DataFrame([{'Stkcd': str(i), 'Date': day, 'market_value': i * 1e6 + j * 1000, 'stock_return': 0.001}
                               for j, day in enumerate(shell_days) for i in range(1, 11)])
    shell_caps = pd.DataFrame([{'Stkcd': str(i), 'cap_month': month, 'market_value': i * 1e6}
                              for month in pd.period_range('2007-01', '2009-04', freq='M') for i in range(1, 11)])
    deals = pd.DataFrame({'deal_id': ['d1'], 'Stkcd': ['1'], 'deal_date': pd.to_datetime(['2008-11-01']),
                          'board_date': pd.to_datetime(['2008-11-01']), 'approval_date': pd.to_datetime(['2009-01-01'])})
    distribution = get_reverse_merger_sizes(deals, shell_caps)
    assert distribution['events']['cap_month'].iloc[0] == pd.Period('2008-08', freq='M')
    assert distribution['distribution'].loc[1, 'fraction'] == 1.0
    values = get_shell_values(deals, shell_daily, shell_caps)['daily']
    assert pd.isna(values.loc[pd.Timestamp('2009-01-05'), 'g'])
    assert values.loc[pd.Timestamp('2009-03-03'), 'g'] > 0
    expected_p = 1 / 3 / 2
    assert np.isclose(values.loc[pd.Timestamp('2009-03-03'), 'p'], expected_p)
    expected_g = values.loc[pd.Timestamp('2009-03-03'), 'g']
    assert np.isclose(values.loc[pd.Timestamp('2009-03-03'), 'shell_value'], expected_p * expected_g / (0.03 + expected_p))
    checks.append({'check': '三个月前分组/已完成G/pG公式', 'passed': True})

    boundary_deals = deals.copy()
    boundary_deals['deal_date'] = pd.to_datetime(['2007-01-01'])
    boundary_caps = pd.DataFrame([{'Stkcd': str(i), 'cap_month': pd.Period('2006-10', freq='M'), 'market_value': i * 1e6} for i in range(1, 11)])
    boundary = get_shell_values(boundary_deals, shell_daily, boundary_caps)['daily']
    assert boundary.loc[pd.Timestamp('2009-01-05'), 'p'] == 0
    assert boundary.loc[pd.Timestamp('2009-01-05'), 'shell_value'] == 0
    checks.append({'check': '730自然日边界/确知零事件', 'passed': True})

    months = pd.period_range('2007-01', periods=24, freq='M')
    rng = np.random.default_rng(76)
    proxy = pd.Series(rng.normal(0, 0.03, len(months)), index=months)
    middle = rng.normal(0, 0.02, len(months))
    largest = rng.normal(0, 0.04, len(months))
    groups = pd.DataFrame({'smallest': 0.01 + 1.7 * proxy + 0.3 * middle - 0.2 * largest,
                           'middle': middle, 'largest': largest}, index=months)
    estimated = get_shell_return_regressions(groups, proxy, 'synthetic')
    assert np.isclose(estimated.loc[estimated['size_group'] == 'smallest', 'b'].iloc[0], 1.7)
    master = pd.DataFrame({'Stkcd': ['a', 'b'], 'listing_date': pd.to_datetime(['2007-01-01', '2007-01-02'])})
    ipo = get_ipo_proxy(master, months)
    assert ipo['nipo'].iloc[0] == 2 and pd.isna(ipo['log_nipo'].iloc[1])
    short_panel = pd.DataFrame({'Stkcd': [str(i) for i in range(1, 11)], 'month': pd.Period('2007-01', freq='M'),
                               'lagged_market_value': np.arange(1, 11, dtype=float), 'stock_return': np.arange(1, 11, dtype=float) / 100})
    weighted = get_size_group_returns(short_panel)
    assert np.isclose(weighted['smallest'].iloc[0], (0.01 + 0.04 + 0.09) / 6)
    rm = get_reverse_merger_returns(deals, shell_daily)
    assert np.isclose(rm.loc[pd.Period('2008-12', freq='M'), 'rm'], 1.001 ** rm.loc[pd.Period('2008-12', freq='M'), 'market_days'] - 1)
    assert pd.isna(rm.loc[pd.Period('2007-01', freq='M'), 'rm'])
    checks.append({'check': 'A2其他组控制/VW/RM复利/零IPO', 'passed': True})

    classifications = pd.DataFrame({'Stkcd': ['a', 'b'], 'formation_date': pd.PeriodIndex(['2000-01', '2000-01'], freq='M'),
                                     'is_financial': [False, False], 'is_real_estate': [False, True]})
    characteristic_sample = classifications[['Stkcd', 'formation_date']].copy()
    nonfinancial, exclusions = get_nonfinancial_characteristics(characteristic_sample, classifications)
    assert nonfinancial['Stkcd'].tolist() == ['a'] and exclusions['excluded'].iloc[0] == 1
    industry_labels = classifications[['Stkcd', 'formation_date']].copy()
    industry_labels['industry37'] = ['industry1', 'industry1']
    industry_panel = characteristic_sample.copy()
    industry_panel['yield_date'] = pd.Period('2000-02', freq='M')
    industry_panel['monthly_market_value'] = [1.0, 3.0]
    industry_panel['monthly_excess_return'] = [0.01, 0.03]
    industry = get_industry37_returns(industry_panel, industry_labels)
    assert np.isclose(industry.iloc[0, 0], 0.025)
    checks.append({'check': '房地产剔除/形成月历史分类/行业VW', 'passed': True})

    long_months = pd.period_range('2000-01', periods=80, freq='M')
    history = pd.DataFrame([{'Stkcd': str(i), 'month': month, 'stock_return': 0.001 * i,
                            'excess_return': 0.001 * i, 'lagged_market_value': float(i), 'eligible': True}
                           for month in long_months for i in range(1, 21)])
    horizon_output = get_reversal_horizon_returns(history)
    assert horizon_output['returns'][60].dropna().index.min() == pd.Period('2005-01', freq='M')
    assert np.isclose(horizon_output['returns'][1].dropna().iloc[0], (0.001 + 0.004)/3 - (0.361 + 0.400)/39)
    missing_history = history[history['month'] != pd.Period('2004-01', freq='M')]
    missing_output = get_reversal_horizon_returns(missing_history, horizons=(60,))
    assert missing_output['returns'][60].notna().sum() == 0
    checks.append({'check': '反转完整月历/60月起始/缺月不压缩', 'passed': True})

    model_input = pd.DataFrame({'multiplier': [1.1, 1.2], 'ch3_log_ml': [-10000, -10000], 'ff3_log_ml': [-10000, -10002]})
    posterior = get_model_posteriors(model_input, 'synthetic arithmetic, not an empirical likelihood')
    assert posterior['ch3_probability'].iloc[0] == 0.5
    assert np.allclose(posterior['ch3_probability'] + posterior['ff3_probability'], 1)
    assert posterior['ch3_probability'].iloc[1] > 0.5
    checks.append({'check': '等先验稳定归一化', 'passed': True})
    return pd.DataFrame(checks)


extension_behaviour_checks = check_extended_analysis_behaviour()
display(extension_behaviour_checks)